In [1]:
import pandas as pd
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk

# -------------------------
# NLTK setup
# -------------------------
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
stop_words = set(stopwords.words('english'))

# -------------------------
# Load CSV
# -------------------------
df = pd.read_csv("D:\\document_references_fhir_raw.csv")  # adjust filename

# -------------------------
# NLP preprocessing
# -------------------------
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)  # keep numbers for labs
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words]
    return ' '.join(tokens)

df['cleaned_text'] = df['note_text'].apply(clean_text)

print("\nSample cleaned text:")
print(df['cleaned_text'].head())

# -------------------------
# CKD stage assignment function
# -------------------------
def assign_ckd_stage(egfr):
    if egfr is None:
        return None
    elif egfr >= 90:
        return 1
    elif 60 <= egfr < 90:
        return 2
    elif 45 <= egfr < 60:
        return '3a'
    elif 30 <= egfr < 45:
        return '3b'
    elif 15 <= egfr < 30:
        return 4
    elif egfr < 15:
        return 5

# -------------------------
# CKD keywords and symptoms
# -------------------------
ckd_terms = ['chronic kidney disease', 'ckd', 'egfr', 'creatinine', 'renal insufficiency', 'kidney failure', 'dialysis']
ckd_symptoms = ['fatigue', 'swelling', 'edema', 'nausea', 'vomiting', 'shortness of breath', 'high blood pressure', 'hypertension']

# -------------------------
# Vectorized search using regex
# -------------------------
df['ckd_mentions'] = df['cleaned_text'].str.contains('|'.join(ckd_terms), case=False, regex=True)
df['ckd_symptoms'] = df['cleaned_text'].str.contains('|'.join(ckd_symptoms), case=False, regex=True)
df['ckd_with_symptoms'] = df['ckd_mentions'] & df['ckd_symptoms']

# -------------------------
# Extract numeric lab values
# -------------------------
def extract_egfr(text):
    match = re.search(r'\b(eGFR|egfr)\s*[:=]?\s*(\d+)', str(text), re.IGNORECASE)
    return int(match.group(2)) if match else None

def extract_creatinine(text):
    match = re.search(r'\b(creatinine|cr)\s*[:=]?\s*(\d+(\.\d+)?)', str(text), re.IGNORECASE)
    return float(match.group(2)) if match else None

df['egfr'] = df['note_text'].apply(extract_egfr)
df['creatinine'] = df['note_text'].apply(extract_creatinine)
df['ckd_stage'] = df['egfr'].apply(assign_ckd_stage)

# -------------------------
# Create CKD Label for ML
# -------------------------
# Binary label: 1 = likely CKD (mentions + symptoms), 0 = no CKD indicators
df['ckd_label'] = df['ckd_with_symptoms'].astype(int)

print("\nLabel distribution:")
print(df['ckd_label'].value_counts())

# -------------------------
# Select relevant columns for ML
# -------------------------
output_w_notes_df = df[['patient_uuid', 'note_text', 'cleaned_text', 
                        'ckd_mentions', 'ckd_symptoms', 'ckd_with_symptoms', 
                        'egfr', 'creatinine', 'ckd_stage', 'ckd_label']]

output_no_notes_df = df[['patient_uuid', 'ckd_mentions', 'ckd_symptoms', 
                         'ckd_with_symptoms', 'egfr', 'creatinine', 
                         'ckd_stage', 'ckd_label']]

# -------------------------
# Save outputs
# -------------------------
output_w_notes_df.to_csv("ckd_nlp_results_with_labs_labeled.csv", index=False)
output_no_notes_df.to_csv("ckd_nlp_results_no_notes_labeled.csv", index=False)

print("\n✅ Files saved with CKD labels ready for machine learning.")
print(output_w_notes_df.head())


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\tonim\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\tonim\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\tonim\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\tonim\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!



Sample cleaned text:
0    2019 06 08 chief complaint frequent urination ...
1    2019 06 05 chief complaint frequent urination ...
2    2019 06 16 chief complaint frequent urination ...
3    2019 06 23 chief complaint frequent urination ...
4    2019 06 30 chief complaint frequent urination ...
Name: cleaned_text, dtype: object

Label distribution:
ckd_label
0    293080
1     57435
Name: count, dtype: int64

✅ Files saved with CKD labels ready for machine learning.
                           patient_uuid  \
0  16ec7045-4e43-cb9e-0445-f43732e3c63c   
1  16ec7045-4e43-cb9e-0445-f43732e3c63c   
2  16ec7045-4e43-cb9e-0445-f43732e3c63c   
3  16ec7045-4e43-cb9e-0445-f43732e3c63c   
4  16ec7045-4e43-cb9e-0445-f43732e3c63c   

                                           note_text  \
0  \n2019-06-08\n\n# Chief Complaint\n- Frequent ...   
1  \n2019-06-05\n\n# Chief Complaint\n- Frequent ...   
2  \n2019-06-16\n\n# Chief Complaint\n- Frequent ...   
3  \n2019-06-23\n\n# Chief Complaint\n- Freque

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# ----------------------------
# 1. Sample dataset
# ----------------------------

df_sample = pd.read_csv("C:\\Users\\tonim\\Documents\\FHIR_project\\ckd_nlp_results_with_labs_labeled.csv")

# ----------------------------
# 2. Define a simple pipeline
# ----------------------------
pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer(
        stop_words='english',
        ngram_range=(1,2)  # capture single and two-word phrases like 'chronic kidney'
    )),
    ('classifier', LogisticRegression())
])

# ----------------------------
# 3. Train/test split
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    df_sample['note_text'], df_sample['ckd_label'], test_size=0.3, random_state=42
)

# ----------------------------
# 4. Fit and evaluate
# ----------------------------
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00     87972
           1       1.00      0.98      0.99     17183

    accuracy                           1.00    105155
   macro avg       1.00      0.99      0.99    105155
weighted avg       1.00      1.00      1.00    105155



In [6]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np

# Features
X = df[['cleaned_text', 'egfr', 'creatinine']]
y = df['ckd_label']

# Replace None with 0 for missing labs
X['egfr'] = X['egfr'].fillna(0)
X['creatinine'] = X['creatinine'].fillna(0)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Column transformer: TF-IDF for text, scaling for numeric
preprocessor = ColumnTransformer([
    ('text', TfidfVectorizer(max_features=1000, stop_words='english'), 'cleaned_text'),
    ('num', StandardScaler(), ['egfr', 'creatinine'])
])

# Pipeline with random forest
model = Pipeline([
    ('preprocess', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Train
model.fit(X_train, y_train)

# Evaluate
print("Model accuracy:", model.score(X_test, y_test))


C:\Users\tonim\AppData\Local\Temp\ipykernel_35600\3939510200.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X['egfr'] = X['egfr'].fillna(0)
C:\Users\tonim\AppData\Local\Temp\ipykernel_35600\3939510200.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['egfr'] = X['egfr'].fillna(0)
C:\Users\tonim\AppData\Local\Temp\ipykernel_35600\3939510200.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To o

Model accuracy: 0.9995720582571359


In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np

# -------------------------
# Load labeled dataset
# -------------------------
df = pd.read_csv("ckd_nlp_results_with_labs_labeled.csv")

# -------------------------
# Clean up and prepare features
# -------------------------
# Replace missing numeric values
df['egfr'] = df['egfr'].fillna(0)
df['creatinine'] = df['creatinine'].fillna(0)

# Select features and target
X = df[['cleaned_text', 'egfr', 'creatinine']]
y = df['ckd_label']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -------------------------
# Build preprocessing pipeline
# -------------------------
preprocessor = ColumnTransformer([
    ('text', TfidfVectorizer(max_features=1000, stop_words='english'), 'cleaned_text'),
    ('num', StandardScaler(), ['egfr', 'creatinine'])
])

# -------------------------
# Combine with classifier
# -------------------------
model = Pipeline([
    ('preprocess', preprocessor),
    ('clf', RandomForestClassifier(
        n_estimators=200, 
        max_depth=10, 
        random_state=42, 
        class_weight='balanced'
    ))
])

# -------------------------
# Train model
# -------------------------
model.fit(X_train, y_train)

# -------------------------
# Evaluate model
# -------------------------
y_pred = model.predict(X_test)

print("\n Model Performance:")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# -------------------------
# Feature importance (optional)
# -------------------------
# Access text + numeric feature importances from the random forest
rf = model.named_steps['clf']
print("\nNumber of text features (TF-IDF):", len(model.named_steps['preprocess']
      .named_transformers_['text'].get_feature_names_out()))

# Save model (optional)
import joblib
joblib.dump(model, "ckd_prediction_pipeline.pkl")

print("\n CKD prediction pipeline saved as ckd_prediction_pipeline.pkl")



 Model Performance:
Accuracy: 0.992

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00     58616
           1       0.95      1.00      0.98     11487

    accuracy                           0.99     70103
   macro avg       0.98      0.99      0.99     70103
weighted avg       0.99      0.99      0.99     70103


Confusion Matrix:
[[58052   564]
 [   17 11470]]

Number of text features (TF-IDF): 1000

 CKD prediction pipeline saved as ckd_prediction_pipeline.pkl
